## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [16]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [15]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_tracing_disabled,
    OpenAIChatCompletionsModel,
)
import os

load_dotenv(override=True)


azure_client = AsyncOpenAI(
    api_key=os.getenv("AZURE_API_KEY"),
    base_url=os.getenv("AZURE_ENDPOINT"),
)

set_default_openai_client(azure_client)
set_tracing_disabled(
    True
)  # OpenAI tracing needs an OpenAI platform key — disable unless you have one


In [ ]:
# Constants 

USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [18]:
from tavily import TavilyClient
from agents import function_tool
import os

client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@function_tool
def web_search(query: str) -> str:
    """Search the web."""
    result = client.search(
        query=query,
        max_results=5,
    )

    return "\n\n".join(r["content"] for r in result["results"])


In [19]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [web_search]


In [20]:
search_agent = Agent(
    name="Search Agent",
    instructions=INSTRUCTIONS,
    tools=tools,
    model=OpenAIChatCompletionsModel(
        model="gpt-chat-latest", openai_client=azure_client
    ),
    model_settings=settings,
)

In [21]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

The AI agent framework ecosystem in 2026 is centered around a handful of mature platforms that balance orchestration, tool use, memory, and production reliability. Across recent industry comparisons, LangGraph is consistently identified as the leading framework for production-grade agent systems, especially for stateful, long-running, and multi-agent workflows. LangChain remains one of the most widely adopted open-source ecosystems overall, often paired with LangGraph for orchestration. CrewAI is widely recommended for quickly building collaborative multi-agent applications, while OpenAI Agents SDK, Google ADK, and Anthropic's Claude Agent SDK have become strong choices for developers building within their respective model ecosystems.

Other notable frameworks include Microsoft Agent Framework (which unifies Semantic Kernel and AutoGen capabilities), LlamaIndex Workflows for data-centric and retrieval-heavy applications, Pydantic AI for strongly typed Python agent development, and Mastra for modern TypeScript-based agent systems. AutoGen (AG2) continues to see use in research and conversational multi-agent scenarios but is increasingly complemented by newer production-focused frameworks. Overall, the market has shifted from experimental agent libraries toward frameworks emphasizing observability, persistence, guardrails, Model Context Protocol (MCP) support, and enterprise deployment, with LangGraph, CrewAI, and the major model-provider SDKs emerging as the most commonly recommended choices.

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [22]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [23]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [24]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(
    name="Planner Agent",
    instructions=INSTRUCTIONS,
    model=OpenAIChatCompletionsModel(model="gpt-5.6-sol", openai_client=azure_client),
    output_type=WebSearchPlan,
)

In [25]:

result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find recent comparative rankings and expert overviews of leading agent frameworks in 2026.', query='most popular AI agent frameworks 2026 comparison LangGraph CrewAI AutoGen OpenAI Agents SDK'), WebSearchItem(reason='Identify frameworks with the strongest open-source adoption using current GitHub activity, stars, contributors, and release cadence.', query='AI agent frameworks GitHub stars contributors activity 2026 LangGraph CrewAI AutoGen Semantic Kernel'), WebSearchItem(reason='Gather popularity and usage evidence from current developer surveys and industry adoption reports.', query='2026 developer survey AI agent framework adoption usage statistics'), WebSearchItem(reason='Assess major vendor-backed frameworks and their enterprise traction, including newer or renamed offerings.', query='top enterprise AI agent frameworks 2026 OpenAI Agents SDK Google ADK Microsoft Agent Framework AWS Strands'), WebSearchItem(reason='Find ecosystem-level 

## Agent 3: The Writer Agent

In [26]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="Writer Agent",
    instructions=INSTRUCTIONS,
    model=OpenAIChatCompletionsModel(model="gpt-5.6-sol", openai_client=azure_client),
    output_type=ReportData,
)

## Agent 4: The email agent

In [27]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [ ]:
send_email_tool.params_json_schema

In [28]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(
    name="Email Agent",
    instructions=INSTRUCTIONS,
    tools=[send_email_tool],
    model=OpenAIChatCompletionsModel(model="gpt-5.6-sol", openai_client=azure_client),
)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [29]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [34]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    print(search_results)
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    
    print(result.final_output)
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [35]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
["Recent expert comparisons in 2026 generally do not agree on a single definitive ranking, but there is broad consensus around a small group of leading AI agent frameworks. Across industry blogs, engineering guides, and developer comparisons, LangGraph (from the LangChain ecosystem), Microsoft's AutoGen, CrewAI, Google's ADK (Agent Development Kit), OpenAI's Agents SDK, and LlamaIndex are the frameworks most frequently highlighted. Rankings tend to vary depending on evaluation criteria such as multi-agent orchestration, production readiness, observability, tool integration, workflow control, and enterprise deployment rather than overall popularity alone.\n\nThe available comparisons consistently describe different strengths rather than declaring an overall winner. LangGraph is commonly recommended for complex, stateful production workflows; AutoGen for multi-agent collaboration

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>